First on private leaderboard

In [2]:
import os
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import defaultdict
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("==========================================================")
print("=== THE 0.7855 RANK 1 MASTERPIECE (DYNAMIC ROUTING) ======")
print("==========================================================")

def seed_everything(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\n--- 1. Loading & Preparing Data ---")
train_baskets = pd.read_csv('train_baskets.csv', dtype={'user_id': str, 'transaction_id': str})
train_transactions = pd.read_csv('train_transactions.csv', dtype={'user_id': str, 'transaction_id': str})
test_instances = pd.read_csv('test_instances.csv', dtype={'user_id': str})

trans_meal_map = train_transactions[['transaction_id', 'meal_period']].drop_duplicates().set_index('transaction_id')['meal_period'].to_dict()
train_baskets['meal_period'] = train_baskets['transaction_id'].map(trans_meal_map).fillna('Unknown')
train_baskets['item_list'] = train_baskets['item_ids'].apply(lambda x: [i for i in str(x).split('|') if i])

train_baskets['timestamp'] = pd.to_datetime(train_baskets['timestamp'])
train_baskets = train_baskets.sort_values(['timestamp', 'user_id']).reset_index(drop=True)

test_instances['current_basket'] = test_instances['current_basket_items'].fillna('').apply(lambda x: [i for i in str(x).split('|') if i])

print("\n--- 2. Chronological Split (85% Train / 15% Val) ---")
split_idx = int(len(train_baskets) * 0.85)
train_df = train_baskets.iloc[:split_idx].copy()
val_df = train_baskets.iloc[split_idx:].copy()

print("\n--- 3. Simulating the Test Set (30% Empty Baskets) ---")
val_instances = []
val_targets = []

np.random.seed(42)
for val_id, row in val_df.iterrows():
    items = row['item_list']
    n = len(items)
    if n == 0: continue

    if n == 1: cutoff = 0
    else:
        rand_val = np.random.rand()
        if rand_val < 0.30: cutoff = 0
        elif rand_val < 0.90: cutoff = 1
        else: cutoff = n - 1

    val_instances.append({
        'val_id': val_id,
        'user_id': row['user_id'],
        'meal_period': row['meal_period'],
        'day_of_week': row['day_of_week'],
        'current_basket': items[:cutoff]
    })
    val_targets.append(items[cutoff])

print("\n--- 4. Building Vocabs & Time-Decayed User History ---")
user_history_dict = {}
user_last_order = {}

for u, group in train_df.groupby('user_id', sort=True):
    scores = defaultdict(float)
    n_orders = len(group)
    for idx, (_, row) in enumerate(group.iterrows()):
        weight = 1.0 + (idx / n_orders)
        for item in row['item_list']:
            scores[item] += weight

    user_history_dict[u] = sorted(scores.keys(), key=lambda x: (-scores[x], x))
    user_last_order[u] = group.iloc[-1]['item_list']

meal_pop_dict = {}
for meal in sorted(train_df['meal_period'].unique().tolist()):
    items_in_meal = [i for group_idx, row in train_df[train_df['meal_period'] == meal].iterrows() for i in row['item_list']]
    meal_pop_dict[meal] = pd.Series(items_in_meal).value_counts().sort_index().sort_values(ascending=False).index.tolist()

all_train_items = [i for items in train_df['item_list'] for i in items]
global_pop = pd.Series(all_train_items).value_counts().sort_index().sort_values(ascending=False).index.tolist()

unique_items = sorted(list(set(all_train_items)))
item2idx = {item: idx+1 for idx, item in enumerate(unique_items)}
item2idx['<PAD>'] = 0
idx2item = {v: k for k, v in item2idx.items()}
vocab_size = len(item2idx)

unique_users = sorted(list(set(train_df['user_id'].unique())))
user2idx = {u: i for i, u in enumerate(unique_users)}
num_users = len(user2idx)

unique_meals = sorted(list(set(train_df['meal_period'].unique().tolist() + test_instances['meal_period'].dropna().unique().tolist())))
meal2idx = {m: i for i, m in enumerate(unique_meals)}
num_meals = len(meal2idx)

days_list = sorted(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])
day2idx = {d: i for i, d in enumerate(days_list)}

print("\n--- 5. All-Pairs Augmentation & Dataloader ---")
augmented_train_data = []
for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Augmenting Data"):
    basket = row['item_list']
    user = row['user_id']
    meal = row['meal_period']
    day = row['day_of_week']
    for target_pos in range(len(basket)):
        target = basket[target_pos]
        context = basket[:target_pos] + basket[target_pos+1:]
        if item2idx.get(target, 0) != 0:
            augmented_train_data.append((user, context, meal, day, target))

class SharedBasketDataset(torch.utils.data.Dataset):
    def __init__(self, data, max_len=20):
        self.data = data
        self.max_len = max_len
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        user, context, meal, day_str, target = self.data[idx]
        encoded = [item2idx.get(i, 0) for i in context][:self.max_len]
        encoded += [0] * (self.max_len - len(encoded))
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(item2idx[target], dtype=torch.long),
            torch.tensor(user2idx.get(user, 0), dtype=torch.long),
            torch.tensor(meal2idx.get(meal, 0), dtype=torch.long),
            torch.tensor(day2idx.get(day_str, 0), dtype=torch.long)
        )

loader = torch.utils.data.DataLoader(SharedBasketDataset(augmented_train_data), batch_size=256, shuffle=True, num_workers=0)

print("\n--- 6. Defining Architectures (UPGRADE: Day Embedding) ---")
class PureTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.item_embedding = nn.Embedding(vocab_size, 64, padding_idx=0)
        self.user_embedding = nn.Embedding(num_users + 1, 64)
        encoder_layer = nn.TransformerEncoderLayer(d_model=64, nhead=4, batch_first=True, dropout=0.2)
        self.transformer = nn.TransformerEncoder(encoder_layer, 2)
        self.fc = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, vocab_size))

    def forward(self, basket, user):
        x = self.transformer(self.item_embedding(basket)).mean(dim=1)
        return self.fc(torch.cat([x, self.user_embedding(user)], dim=1))

class ContextTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.item_embedding = nn.Embedding(vocab_size, 64, padding_idx=0)
        self.user_embedding = nn.Embedding(num_users + 1, 64)
        self.meal_embedding = nn.Embedding(num_meals, 64)
        self.day_embedding = nn.Embedding(7, 64)
        encoder_layer = nn.TransformerEncoderLayer(d_model=64, nhead=4, batch_first=True, dropout=0.2)
        self.transformer = nn.TransformerEncoder(encoder_layer, 2)
        self.fc = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, vocab_size))

    def forward(self, basket, user, meal, day):
        x = self.transformer(self.item_embedding(basket)).mean(dim=1)
        return self.fc(torch.cat([x, self.user_embedding(user), self.meal_embedding(meal), self.day_embedding(day)], dim=1))

print("\n--- 7. DETERMINISTIC MULTI-SEED TRAINING ---")
SEEDS = [42, 2024, 777, 888, 999, 1234, 5678]
val_seed_predictions = []
test_seed_predictions = []
EPOCHS = 15
RRF_K = 20

for seed_idx, seed in enumerate(SEEDS):
    print(f"\n=============================================")
    print(f"=== TRAINING SEED {seed_idx + 1}/{len(SEEDS)} (Seed: {seed}) ===")
    print(f"=============================================")

    seed_everything(seed)

    model_pure = PureTransformer().to(device)
    model_context = ContextTransformer().to(device)

    opt_pure = optim.AdamW(model_pure.parameters(), lr=0.001, weight_decay=1e-4)
    opt_context = optim.AdamW(model_context.parameters(), lr=0.001, weight_decay=1e-4)

    sched_pure = optim.lr_scheduler.CosineAnnealingLR(opt_pure, T_max=EPOCHS, eta_min=1e-5)
    sched_context = optim.lr_scheduler.CosineAnnealingLR(opt_context, T_max=EPOCHS, eta_min=1e-5)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

    for epoch in range(EPOCHS):
        model_pure.train(); model_context.train()
        for basket, target, user, meal, day in tqdm(loader, desc=f"Seed {seed} | Epoch {epoch+1}/{EPOCHS}"):
            basket, target = basket.to(device), target.to(device)
            user, meal, day = user.to(device), meal.to(device), day.to(device)

            opt_pure.zero_grad()
            l_p = criterion(model_pure(basket, user), target)
            l_p.backward()
            opt_pure.step()

            opt_context.zero_grad()
            l_c = criterion(model_context(basket, user, meal, day), target)
            l_c.backward()
            opt_context.step()

        sched_pure.step(); sched_context.step()

    model_pure.eval(); model_context.eval()

    print(f"Generating Test Preds (Seed {seed})...")
    seed_test_results = {}
    for _, row in tqdm(test_instances.sort_values('instance_id').iterrows(), total=len(test_instances), desc="Test"):
        user, basket, meal, day = row['user_id'], row['current_basket'], row['meal_period'], row['day_of_week']
        inst_id = row['instance_id']

        encoded = [item2idx.get(i, 0) for i in basket][:20]
        encoded += [0] * (20 - len(encoded))

        basket_tensor = torch.tensor([encoded], dtype=torch.long).to(device)
        user_tensor = torch.tensor([user2idx.get(user, num_users)], dtype=torch.long).to(device)
        meal_tensor = torch.tensor([meal2idx.get(meal, 0)], dtype=torch.long).to(device)
        day_tensor = torch.tensor([day2idx.get(day, 0)], dtype=torch.long).to(device)

        with torch.no_grad():
            out_p = model_pure(basket_tensor, user_tensor)
            out_c = model_context(basket_tensor, user_tensor, meal_tensor, day_tensor)
            for item_idx in encoded:
                if item_idx != 0:
                    out_p[0, item_idx] = -float('inf')
                    out_c[0, item_idx] = -float('inf')

            topk_p = torch.topk(out_p, 20).indices.cpu().numpy()[0]
            topk_c = torch.topk(out_c, 20).indices.cpu().numpy()[0]

        # The specific Dynamic Neural Routing that made this model great
        is_empty = (len(basket) == 0)
        w_p = 0.5 if is_empty else 1.5
        w_c = 1.5 if is_empty else 1.0

        scores = defaultdict(float)
        for i, item_idx in enumerate(topk_p):
            if item_idx in idx2item: scores[idx2item[item_idx]] += w_p / (RRF_K + i + 1)
        for i, item_idx in enumerate(topk_c):
            if item_idx in idx2item: scores[idx2item[item_idx]] += w_c / (RRF_K + i + 1)

        seed_test_results[inst_id] = sorted(scores.keys(), key=lambda x: (scores[x], x), reverse=True)
    test_seed_predictions.append(seed_test_results)


print("\n--- 8. GENERATING EXACT REPRODUCIBLE LEADERBOARD SUBMISSION ---")
final_submission = []
for _, row in test_instances.sort_values('instance_id').iterrows():
    inst_id = row['instance_id']
    user = row['user_id']
    basket = row['current_basket']
    meal = row['meal_period']

    master_scores = defaultdict(float)
    for seed_preds in test_seed_predictions:
        for i, item in enumerate(seed_preds[inst_id]):
            master_scores[item] += 1.0 / (RRF_K + i + 1)

    ensemble_preds = sorted(master_scores.keys(), key=lambda x: (master_scores[x], x), reverse=True)
    final_top_10 = [i for i in ensemble_preds if i not in basket][:10]

    # Fallbacks to ensure exactly 10 items
    if len(final_top_10) < 10:
        for p_item in user_last_order.get(user, []):
            if p_item not in final_top_10 and p_item not in basket: final_top_10.append(p_item)
            if len(final_top_10) == 10: break

    if len(final_top_10) < 10:
        for p_item in user_history_dict.get(user, []):
            if p_item not in final_top_10 and p_item not in basket: final_top_10.append(p_item)
            if len(final_top_10) == 10: break

    if len(final_top_10) < 10:
        for p_item in meal_pop_dict.get(meal, global_pop):
            if p_item not in final_top_10 and p_item not in basket: final_top_10.append(p_item)
            if len(final_top_10) == 10: break

    final_submission.append([inst_id] + final_top_10)

columns = ['instance_id'] + [f'rank_{i}' for i in range(1, 11)]
pd.DataFrame(final_submission, columns=columns).to_csv('RANK_1_FINAL_SUBMISSION.csv', index=False)

print("\n--- 9. RUNNING FINAL CLEANUP FILTER ---")
menu_items = pd.read_csv('menu_items.csv')
valid_items = set(menu_items['item_id'].dropna().astype(str).tolist())
sub = pd.read_csv('RANK_1_FINAL_SUBMISSION.csv')

cleaned_rows = []
for _, row in sub.iterrows():
    instance_id = row['instance_id']
    raw_preds = [str(row[f'rank_{i}']).strip() for i in range(1, 11)]
    valid_preds = []

    for p in raw_preds:
        if p in valid_items and p not in valid_preds: valid_preds.append(p)

    while len(valid_preds) < 10:
        for safe_item in [i for i in global_pop if i in valid_items]:
            if safe_item not in valid_preds: valid_preds.append(safe_item)
            if len(valid_preds) == 10: break

    cleaned_rows.append([instance_id] + valid_preds)

pd.DataFrame(cleaned_rows, columns=columns).to_csv('RANK_5.csv', index=False)
print("✅ Fully Cleaned and Ready! Generated 'RANK_5.csv'")

=== THE 0.7855 RANK 1 MASTERPIECE (DYNAMIC ROUTING) ======

--- 1. Loading & Preparing Data ---

--- 2. Chronological Split (85% Train / 15% Val) ---

--- 3. Simulating the Test Set (30% Empty Baskets) ---

--- 4. Building Vocabs & Time-Decayed User History ---

--- 5. All-Pairs Augmentation & Dataloader ---


Augmenting Data: 100%|██████████| 72663/72663 [00:03<00:00, 19525.50it/s]



--- 6. Defining Architectures (UPGRADE: Day Embedding) ---

--- 7. DETERMINISTIC MULTI-SEED TRAINING ---

=== TRAINING SEED 1/7 (Seed: 42) ===


Seed 42 | Epoch 1/15: 100%|██████████| 568/568 [00:16<00:00, 35.01it/s]
Seed 42 | Epoch 2/15: 100%|██████████| 568/568 [00:14<00:00, 38.17it/s]
Seed 42 | Epoch 3/15: 100%|██████████| 568/568 [00:15<00:00, 36.89it/s]
Seed 42 | Epoch 4/15: 100%|██████████| 568/568 [00:15<00:00, 36.38it/s]
Seed 42 | Epoch 5/15: 100%|██████████| 568/568 [00:15<00:00, 37.00it/s]
Seed 42 | Epoch 6/15: 100%|██████████| 568/568 [00:16<00:00, 35.03it/s]
Seed 42 | Epoch 7/15: 100%|██████████| 568/568 [00:15<00:00, 35.87it/s]
Seed 42 | Epoch 8/15: 100%|██████████| 568/568 [00:16<00:00, 35.30it/s]
Seed 42 | Epoch 9/15: 100%|██████████| 568/568 [00:15<00:00, 36.49it/s]
Seed 42 | Epoch 10/15: 100%|██████████| 568/568 [00:15<00:00, 36.07it/s]
Seed 42 | Epoch 11/15: 100%|██████████| 568/568 [00:16<00:00, 35.40it/s]
Seed 42 | Epoch 12/15: 100%|██████████| 568/568 [00:15<00:00, 36.55it/s]
Seed 42 | Epoch 13/15: 100%|██████████| 568/568 [00:15<00:00, 35.62it/s]
Seed 42 | Epoch 14/15: 100%|██████████| 568/568 [00:15<00:00

Generating Test Preds (Seed 42)...


...


=== TRAINING SEED 7/7 (Seed: 5678) ===


Seed 5678 | Epoch 1/15: 100%|██████████| 568/568 [00:15<00:00, 35.52it/s]
Seed 5678 | Epoch 2/15: 100%|██████████| 568/568 [00:15<00:00, 35.65it/s]
Seed 5678 | Epoch 3/15: 100%|██████████| 568/568 [00:15<00:00, 35.64it/s]
Seed 5678 | Epoch 4/15: 100%|██████████| 568/568 [00:15<00:00, 35.77it/s]
Seed 5678 | Epoch 5/15: 100%|██████████| 568/568 [00:15<00:00, 36.34it/s]
Seed 5678 | Epoch 6/15: 100%|██████████| 568/568 [00:15<00:00, 35.86it/s]
Seed 5678 | Epoch 7/15: 100%|██████████| 568/568 [00:15<00:00, 36.13it/s]
Seed 5678 | Epoch 8/15: 100%|██████████| 568/568 [00:16<00:00, 35.23it/s]
Seed 5678 | Epoch 9/15: 100%|██████████| 568/568 [00:15<00:00, 36.30it/s]
Seed 5678 | Epoch 10/15: 100%|██████████| 568/568 [00:15<00:00, 36.05it/s]
Seed 5678 | Epoch 11/15: 100%|██████████| 568/568 [00:15<00:00, 35.89it/s]
Seed 5678 | Epoch 12/15: 100%|██████████| 568/568 [00:15<00:00, 36.25it/s]
Seed 5678 | Epoch 13/15: 100%|██████████| 568/568 [00:15<00:00, 35.80it/s]
Seed 5678 | Epoch 14/15: 100%|████

Generating Test Preds (Seed 5678)...


Test: 100%|██████████| 14147/14147 [00:39<00:00, 362.54it/s]



--- 8. GENERATING EXACT REPRODUCIBLE LEADERBOARD SUBMISSION ---

--- 9. RUNNING FINAL CLEANUP FILTER ---
✅ Fully Cleaned and Ready! Generated 'RANK_5.csv'


In [5]:
import pandas as pd

# Load both files
file1 = pd.read_csv('RANK_5.csv')
file2 = pd.read_csv('submission_4c8256ea-33f.csv')

# Check if every single row and column matches exactly
are_identical = file1.equals(file2)
print(f"Are the files exactly the same? {are_identical}")

Are the files exactly the same? True
